# Zerobus Ingest Benchmark Driver

Run all ingest modes (gRPC sync/async, HTTP/1.1 sync/async, HTTP/2 sync/async)
across a configurable number of iterations using `zbhelper.ingest_v2`.

Select which modes to run via the **widgets in Step 3**, then run all cells top to bottom.
Results are appended to `benchmark_results.jsonl` and displayed as a flattened table.

### Step 1: Install dependencies

In [1]:
%pip install --quiet databricks-sdk[notebook] databricks-zerobus-ingest-sdk aiohttp requests httpx[http2]

Note: you may need to restart the kernel to use updated packages.


In [2]:
# DO NOT add %autoreload here — it resets the module-level OAuth token cache
# (_token_cache in ingest_v2.py) on every cell execution, forcing a token
# re-fetch every run.  If you edit zbhelper, restart the kernel manually
# (Kernel → Restart) and re-run from this cell.

import sys
from pathlib import Path

_nb = next((str(p) for p in [Path.cwd(), Path.cwd() / "notebooks"] if (p / "zbhelper").is_dir()), None)
if _nb and _nb not in sys.path:
    sys.path.insert(0, _nb)

import zbhelper.ingest_v2 as zbv2
import zbhelper.setup as zbsetup
print("zbhelper loaded")

zbhelper loaded


### Step 2: Connection & table configuration

`zbhelper.setup` auto-discovers the workspace URL, region, and ZeroBus endpoint, then
ensures the service principal and OAuth secret exist. Set `SP_NAME` below — everything
else is derived automatically.

In [3]:
# ── User-configurable ────────────────────────────────────────────────────
SP_NAME = "lfcdemo_zerobus"   # service principal display name (drives secret scope)

# ── Step 2.a: auto-discover workspace + ZeroBus endpoint ─────────────────
_ws = zbsetup.discover_workspace(dbutils)
DATABRICKS_WORKSPACE_URL = _ws["workspace_url"]
DATABRICKS_WORKSPACE_ID  = _ws["workspace_id"]
ZEROBUS_INGEST_URL       = _ws["zerobus_ingest_url"]
SERVER_ENDPOINT          = _ws["server_endpoint"]

# ── Step 2.b + 2.c: ensure SP exists; mint/validate OAuth secret ──────────
_sp = zbsetup.ensure_service_principal(_ws, dbutils, sp_name=SP_NAME)
CLIENT_ID     = _sp["client_id"]
CLIENT_SECRET = _sp["client_secret"]

Cleared metadata-service auth env vars (Connect kernel).
workspace_url=https://e2-demo-field-eng.cloud.databricks.com
workspace_id=1444828305810485
region=us-west-2
server_endpoint=https://1444828305810485.zerobus.us-west-2.cloud.databricks.com
SP exists: 'lfcdemo_zerobus_sp' sp_id=75332893425169
OAuth client secret is valid.


### Step 3: Benchmark widgets

Select which modes, concurrency levels, and iteration count to run.
Defaults run **all 6 modes** × concurrency 1 × 10 iterations.

In [4]:
# ── Canonical option sets — only place to add/remove values ──────────────
_APIS_ALL          = ["grpc", "http/1.1", "http/2"]
_SYNC_ASYNCS_ALL   = ["sync", "async"]
_CONCURRENCIES_ALL = ["1", "2", "4", "8", "16", "32"]

# On Databricks: "all" selects every option; single-value default only (ipywidgets limitation)
dbutils.widgets.multiselect("api",         "all",  ["all"] + _APIS_ALL)
dbutils.widgets.multiselect("sync_async",  "all",  ["all"] + _SYNC_ASYNCS_ALL)
dbutils.widgets.multiselect("concurrency", "all",    ["all"] + _CONCURRENCIES_ALL)
dbutils.widgets.dropdown(   "iters",       "10",   ["1","5","10","20","50"])
dbutils.widgets.dropdown(   "n",           "1000", ["100","500","1000","5000"])

Box(children=(Label(value='api'), SelectMultiple(index=(1,), options=(('__EMPTY__', ''), ('all', 'all'), ('grp…

Box(children=(Label(value='sync_async'), SelectMultiple(index=(1,), options=(('__EMPTY__', ''), ('all', 'all')…

Box(children=(Label(value='concurrency'), SelectMultiple(index=(1,), options=(('__EMPTY__', ''), ('all', 'all'…

Box(children=(Label(value='iters'), Dropdown(index=2, options=('1', '5', '10', '20', '50'), value='10')))

Box(children=(Label(value='n'), Dropdown(index=2, options=('100', '500', '1000', '5000'), value='1000')))

In [5]:
def _sel(name, all_vals):
    """Return all_vals if widget is unset or "all", else the selected subset."""
    v = dbutils.widgets.get(name).split(",")
    return all_vals if "all" in v else v

_APIS          = _sel("api",         _APIS_ALL)
_SYNC_ASYNCS   = _sel("sync_async",  _SYNC_ASYNCS_ALL)
_CONCURRENCIES = _sel("concurrency", _CONCURRENCIES_ALL)
_ITERS         = int(dbutils.widgets.get("iters"))
_N             = int(dbutils.widgets.get("n"))

_MODES = []
for _a in _APIS:
    for _s in _SYNC_ASYNCS:
        _m = zbv2.MODE_MAP.get((_a, _s))
        if _m:
            _MODES.append(_m)
        else:
            print(f"Warning: unsupported combination api={_a!r} sync_async={_s!r} — skipped")

print(f"Modes to run : {_MODES}")
print(f"Concurrencies: {_CONCURRENCIES}")
print(f"Iters × N    : {_ITERS} × {_N}")
print(f"Total runs   : {len(_MODES) * len(_CONCURRENCIES) * _ITERS}")

Modes to run : ['grpc_sync', 'grpc_async', 'http_sync', 'http_async', 'http2_sync', 'http2_async']
Concurrencies: ['1', '2', '4', '8', '16', '32']
Iters × N    : 10 × 1000
Total runs   : 360


### Step 4: Create tables

One table per mode (`airquality_<mode>`). Tables are created if they don't exist and
the service principal is granted the required UC privileges.

In [6]:
# Create one UC table per selected mode; cache catalog/schema for the cfg dict.
_tables: dict[str, dict] = {}
for _m in _MODES:
    _tbl = zbsetup.ensure_table(spark, _ws, CLIENT_ID, table=f"airquality_{_m}")
    _tables[_m] = _tbl
    print(f"  {_m:15s} → {_tbl['table_name']}")

CATALOG = _tables[_MODES[0]]["catalog"]
SCHEMA  = _tables[_MODES[0]]["schema"]
print(f"\nCATALOG={CATALOG!r}  SCHEMA={SCHEMA!r}")

CATALOG auto-detected: 'main'
SCHEMA auto-detected: 'robert_lee'
Table ready: main.robert_lee.airquality_grpc_sync
  grpc_sync       → main.robert_lee.airquality_grpc_sync


CATALOG auto-detected: 'main'
SCHEMA auto-detected: 'robert_lee'
Table ready: main.robert_lee.airquality_grpc_async
  grpc_async      → main.robert_lee.airquality_grpc_async


CATALOG auto-detected: 'main'
SCHEMA auto-detected: 'robert_lee'
Table ready: main.robert_lee.airquality_http_sync
  http_sync       → main.robert_lee.airquality_http_sync


CATALOG auto-detected: 'main'
SCHEMA auto-detected: 'robert_lee'
Table ready: main.robert_lee.airquality_http_async
  http_async      → main.robert_lee.airquality_http_async


CATALOG auto-detected: 'main'
SCHEMA auto-detected: 'robert_lee'
Table ready: main.robert_lee.airquality_http2_sync
  http2_sync      → main.robert_lee.airquality_http2_sync


CATALOG auto-detected: 'main'
SCHEMA auto-detected: 'robert_lee'
Table ready: main.robert_lee.airquality_http2_async
  http2_async     → main.robert_lee.airquality_http2_async

CATALOG='main'  SCHEMA='robert_lee'


### Step 5: Benchmark loop

**Once before the loop:**
- Network ping — TCP SYN/ACK to host:443 (`ping_ms`, no TLS or HTTP)

**Once per mode (first time that mode is seen):**
- HTTP ping — warm GET on the open session, no insert (`http_ping_ms`, HTTP modes only)
- OAuth token fetch — cached for all subsequent iterations of the same mode

**For each (mode × concurrency × iteration):**
- Sync modes (`grpc_sync`, `http_sync`, `http2_sync`) with `concurrency > 1` are **skipped** — concurrency is meaningless for blocking calls.
1. Opens a stream / HTTP session (`setup_zerobus`; reuses cached OAuth token)
2. **4a** — launches all `min(10, N)` single-row inserts via `asyncio.gather`; semaphore caps in-flight calls at `concurrency`
3. **4b** — launches all 10 batch runs via `asyncio.gather`; same semaphore caps in-flight calls at `concurrency`
4. Closes the session and polls UC visibility
5. Appends the result to `benchmark_results.jsonl`

In [7]:
import asyncio as _asyncio
import datetime
import time

from pathlib import Path
from statistics import mean, median

_RESULTS_FILE = Path(".") / "benchmark_results.jsonl"
_BATCH_RUNS   = 10   # number of batch runs per iteration (matches zerobus_grpc_http.ipynb)

# ── one-time baseline measurements ───────────────────────────────────────────
_ping_s = zbv2.ping_endpoint(ZEROBUS_INGEST_URL)
print(f"Network ping: {_ping_s * 1000:.1f} ms")

_mode_http_ping_s: dict[str, float | None] = {}  # measured once per mode

for _iter in range(1, _ITERS + 1):
    for _mode in _MODES:
        _is_async = _mode.endswith("_async")
        _concurrencies = [int(c) for c in _CONCURRENCIES]
        for _concurrency in _concurrencies:
            if _concurrency > 1 and not _is_async:
                print(f"skip: {_mode} is sync — concurrency={_concurrency} has no effect")
                continue
            print(f"\n{'='*64}")
            print(f"iter={_iter}/{_ITERS}  mode={_mode}  concurrency={_concurrency}")
            print(f"{'='*64}")

            _tbl        = _tables[_mode]
            _table_name = _tbl["table_name"]

            _cfg = {
                "server_endpoint":    SERVER_ENDPOINT,
                "workspace_url":      DATABRICKS_WORKSPACE_URL,
                "workspace_id":       DATABRICKS_WORKSPACE_ID,
                "zerobus_ingest_url": ZEROBUS_INGEST_URL,
                "client_id":          CLIENT_ID,
                "client_secret":      CLIENT_SECRET,
                "catalog":            _tbl["catalog"],
                "schema":             _tbl["schema"],
                "table_name":         _table_name,
            }

            # ── build records ─────────────────────────────────────────────
            _singles_n = min(10, _N)
            _batch_n   = _N - _singles_n
            _records_4a    = zbv2.build_records(_singles_n)
            _batch_records = zbv2.build_records(_batch_n, offset=_singles_n)

            # ── baseline ──────────────────────────────────────────────────
            _baseline = zbv2.fetch_row_baseline(spark, _table_name)
            _row_before = _baseline["count"]

            # ── setup ─────────────────────────────────────────────────────
            _zb_client = await zbv2.setup_zerobus(_mode, _cfg)
            _connect_s = _zb_client["_connect_s"]
            _oauth_s   = _zb_client["_oauth_s"]

            # ── HTTP ping (once per mode for entire run, HTTP modes only) ──
            if _mode not in _mode_http_ping_s:
                _http_ping_s = await zbv2.http_ping(_zb_client, ZEROBUS_INGEST_URL)
                _mode_http_ping_s[_mode] = _http_ping_s
                if _http_ping_s is not None:
                    print(f"HTTP ping ({_mode}): {_http_ping_s * 1000:.1f} ms")
            else:
                _http_ping_s = _mode_http_ping_s[_mode]

            # ── 4a: single-row inserts ────────────────────────────────────
            _row_send_s: list[float] = []
            _row_wait_s: list[float] = []
            _row_ack_s:  list[float] = []

            _sem = _asyncio.Semaphore(_concurrency)

            async def _insert_one(rec):
                async with _sem:
                    return await zbv2.call_zerobus_insert(_zb_client, [rec])

            _t_4a0 = time.perf_counter()
            _res_4a = await _asyncio.gather(*[_insert_one(r) for r in _records_4a])
            _singles_wall_s = time.perf_counter() - _t_4a0
            for _s, _w in _res_4a:
                _row_send_s.append(_s)
                _row_wait_s.append(_w)
                _row_ack_s.append(_s + _w)

            # ── 4b: batch inserts ─────────────────────────────────────────
            _batch_send_s: list[float] = []
            _batch_wait_s: list[float] = []
            _batch_run_s:  list[float] = []
            _disconnect_s  = 0.0
            _t_after_close = time.perf_counter()

            async def _insert_batch():
                async with _sem:
                    return await zbv2.call_zerobus_insert(_zb_client, _batch_records)

            try:
                _res_4b = await _asyncio.gather(*[_insert_batch() for _ in range(_BATCH_RUNS)])
                for _s, _w in _res_4b:
                    _batch_send_s.append(_s)
                    _batch_wait_s.append(_w)
                    _batch_run_s.append(_s + _w)
            finally:
                _tc = time.perf_counter()
                await _zb_client["close"]()
                _t_after_close = time.perf_counter()
                _disconnect_s  = _t_after_close - _tc

            _ingest_4a4b_s = _singles_wall_s + sum(_batch_run_s)
            _total_rows    = _singles_n + _batch_n * len(_batch_run_s)

            # ── 4c: visibility ────────────────────────────────────────────
            _vis = zbv2.poll_visibility(
                spark, _table_name, _row_before + _total_rows,
                _t_after_close, _t_4a0,
            )
            _visibility_s                  = _vis["visibility_s"]
            _visibility_from_first_send_s  = _vis["visibility_from_first_send_s"]

            # ── 4d: print metrics ─────────────────────────────────────────
            print(f"\nIngested {_total_rows} rows → {_table_name}  [mode={_mode}]")
            print(f"  visibility: {_visibility_from_first_send_s*1000:.1f} ms (from first send → COUNT(*) target)")
            print(f"  visibility: {_visibility_s*1000:.1f} ms (from end of ingest → COUNT(*) target)")
            if _oauth_s is not None:
                print(f"  oauth:      {_oauth_s*1000:.1f} ms")
            print(f"  connect:    {_connect_s*1000:.1f} ms")
            print(f"  disconnect: {_disconnect_s*1000:.1f} ms")
            print(f"  ingest wall (4a+4b): {_ingest_4a4b_s*1000:.1f} ms")
            if _row_ack_s:
                print(f"  4a ({_singles_n} singles) wall {_singles_wall_s*1000:.1f} ms"
                      f"  send+wait: min={min(_row_ack_s)*1000:.1f}  "
                      f"median={median(_row_ack_s)*1000:.1f}  max={max(_row_ack_s)*1000:.1f} ms")
            if _batch_run_s:
                print(f"  4b ({_batch_n} rows × {len(_batch_run_s)} runs)"
                      f"  wall min={min(_batch_run_s)*1000:.1f}  "
                      f"median={median(_batch_run_s)*1000:.1f}  max={max(_batch_run_s)*1000:.1f} ms")

            # ── 4e: append result ─────────────────────────────────────────
            _result = {
                "run_at":           datetime.datetime.now(datetime.timezone.utc).isoformat(),
                "mode":             _mode,
                "mode_label":       zbv2.MODE_LABEL.get(_mode, _mode),
                "table":            _table_name,
                "zerobus_endpoint": ZEROBUS_INGEST_URL,
                "n":                _N,
                "concurrency":      _concurrency,
                "oauth_ms":         zbv2.ms(_oauth_s)      if _oauth_s      is not None else None,
                "ping_ms":          zbv2.ms(_ping_s),
                "http_ping_ms":  zbv2.ms(_http_ping_s) if _http_ping_s is not None else None,
                "connect_ms":       zbv2.ms(_connect_s)    if _connect_s    is not None else None,
                "disconnect_ms":    zbv2.ms(_disconnect_s) if _disconnect_s is not None else None,
                "ingest_wall_ms":   zbv2.ms(_ingest_4a4b_s),
                "visibility_from_first_send_ms": zbv2.ms(_visibility_from_first_send_s),
                "visibility_from_end_ms":        zbv2.ms(_visibility_s),
                "4a": {
                    "rows":         1,
                    "runs":         _singles_n,
                    "wall_ms":      zbv2.ms(_singles_wall_s),
                    "send_ms":      zbv2.stats_ms(_row_send_s),
                    "wait_ms":      zbv2.stats_ms(_row_wait_s),
                    "send_wait_ms": zbv2.stats_ms(_row_ack_s),
                } if _row_ack_s else None,
                "4b": {
                    "rows":         _batch_n,
                    "runs":         len(_batch_run_s),
                    "wall_ms":      zbv2.ms(sum(_batch_run_s)),
                    "send_wait_ms": zbv2.stats_ms(_batch_run_s),
                    "send_ms":      zbv2.stats_ms(_batch_send_s),
                    "wait_ms":      zbv2.stats_ms(_batch_wait_s),
                } if _batch_run_s else None,
            }
            zbv2.append_jsonl(_result, _RESULTS_FILE)
            print(f"  → appended to {_RESULTS_FILE}")

Network ping: 196.6 ms

iter=1/10  mode=grpc_sync  concurrency=1


2026-04-21T22:02:24.217159Z  INFO databricks_zerobus_ingest_sdk: Successfully created stream stream_id=2aba23f7-a0fa-4a5a-86c0-383fed0e7d92
2026-04-21T22:02:24.217214Z  INFO databricks_zerobus_ingest_sdk: Successfully created stream stream_id=2aba23f7-a0fa-4a5a-86c0-383fed0e7d92
2026-04-21T22:02:24.217297Z  INFO databricks_zerobus_ingest_sdk: Successfully created new ephemeral stream stream_id=2aba23f7-a0fa-4a5a-86c0-383fed0e7d92
2026-04-21T22:02:24.218418Z  INFO databricks_zerobus_ingest_sdk: Stream is not caught up to any offset yet. Waiting for the first offset. stream_id=2aba23f7-a0fa-4a5a-86c0-383fed0e7d92
[ack callback] offset 0 acknowledged2026-04-21T22:02:24.419971Z  INFO databricks_zerobus_ingest_sdk: Stream is caught up to the given offset. Waiting for acknowledgement completed. stream_id=2aba23f7-a0fa-4a5a-86c0-383fed0e7d92

2026-04-21T22:02:24.421225Z  INFO databricks_zerobus_ingest_sdk: Stream is caught up to offset 0. Waiting for offset 1. stream_id=2aba23f7-a0fa-4a5a-86c


Ingested 9910 rows → main.robert_lee.airquality_grpc_sync  [mode=grpc_sync]
  visibility: 11197.8 ms (from first send → COUNT(*) target)
  visibility: 5965.5 ms (from end of ingest → COUNT(*) target)
  connect:    2010.9 ms
  disconnect: 0.3 ms
  ingest wall (4a+4b): 5230.6 ms
  4a (10 singles) wall 2014.1 ms  send+wait: min=197.7  median=201.4  max=203.2 ms
  4b (990 rows × 10 runs)  wall min=199.7  median=400.9  max=405.0 ms
  → appended to benchmark_results.jsonl
skip: grpc_sync is sync — concurrency=2 has no effect
skip: grpc_sync is sync — concurrency=4 has no effect
skip: grpc_sync is sync — concurrency=8 has no effect
skip: grpc_sync is sync — concurrency=16 has no effect
skip: grpc_sync is sync — concurrency=32 has no effect

iter=1/10  mode=grpc_async  concurrency=1


2026-04-21T22:02:39.609069Z  INFO databricks_zerobus_ingest_sdk: Successfully created stream stream_id=ace44ff7-e796-49c2-8d91-1f0aeb0c3db6
2026-04-21T22:02:39.609142Z  INFO databricks_zerobus_ingest_sdk: Successfully created stream stream_id=ace44ff7-e796-49c2-8d91-1f0aeb0c3db6
2026-04-21T22:02:39.609229Z  INFO databricks_zerobus_ingest_sdk: Successfully created new ephemeral stream stream_id=ace44ff7-e796-49c2-8d91-1f0aeb0c3db6
2026-04-21T22:02:39.611656Z  INFO databricks_zerobus_ingest_sdk: Stream is not caught up to any offset yet. Waiting for the first offset. stream_id=ace44ff7-e796-49c2-8d91-1f0aeb0c3db6
[ack callback] offset 0 acknowledged2026-04-21T22:02:39.812767Z  INFO databricks_zerobus_ingest_sdk: Stream is caught up to the given offset. Waiting for acknowledgement completed. stream_id=ace44ff7-e796-49c2-8d91-1f0aeb0c3db6

2026-04-21T22:02:39.814913Z  INFO databricks_zerobus_ingest_sdk: Stream is caught up to offset 0. Waiting for offset 1. stream_id=ace44ff7-e796-49c2-8d9


Ingested 9910 rows → main.robert_lee.airquality_grpc_async  [mode=grpc_async]
  visibility: 5863.2 ms (from first send → COUNT(*) target)
  visibility: 1637.5 ms (from end of ingest → COUNT(*) target)
  connect:    1472.1 ms
  disconnect: 0.7 ms
  ingest wall (4a+4b): 4224.1 ms
  4a (10 singles) wall 2010.9 ms  send+wait: min=197.8  median=201.4  max=204.3 ms
  4b (990 rows × 10 runs)  wall min=198.9  median=201.3  max=402.4 ms
  → appended to benchmark_results.jsonl

iter=1/10  mode=grpc_async  concurrency=2


...


Ingested 9910 rows → main.robert_lee.airquality_http2_async  [mode=http2_async]
  visibility: 7172.1 ms (from first send → COUNT(*) target)
  visibility: 5925.2 ms (from end of ingest → COUNT(*) target)
  oauth:      0.0 ms
  connect:    323.8 ms
  disconnect: 0.2 ms
  ingest wall (4a+4b): 5706.5 ms
  4a (10 singles) wall 442.6 ms  send+wait: min=196.9  median=249.5  max=252.5 ms
  4b (990 rows × 10 runs)  wall min=194.4  median=608.6  max=611.1 ms
  → appended to benchmark_results.jsonl

iter=10/10  mode=http2_async  concurrency=16


OAuth token reused from cache
[http2_async] TCP+TLS+h2 connected in 329.2 ms



Ingested 9910 rows → main.robert_lee.airquality_http2_async  [mode=http2_async]
  visibility: 1920.7 ms (from first send → COUNT(*) target)
  visibility: 806.1 ms (from end of ingest → COUNT(*) target)
  oauth:      0.0 ms
  connect:    329.2 ms
  disconnect: 0.3 ms
  ingest wall (4a+4b): 7711.1 ms
  4a (10 singles) wall 307.1 ms  send+wait: min=302.4  median=302.7  max=303.1 ms
  4b (990 rows × 10 runs)  wall min=600.3  median=799.8  max=801.1 ms
  → appended to benchmark_results.jsonl

iter=10/10  mode=http2_async  concurrency=32


OAuth token reused from cache
[http2_async] TCP+TLS+h2 connected in 328.8 ms



Ingested 9910 rows → main.robert_lee.airquality_http2_async  [mode=http2_async]
  visibility: 2490.4 ms (from first send → COUNT(*) target)
  visibility: 1583.8 ms (from end of ingest → COUNT(*) target)
  oauth:      0.0 ms
  connect:    328.8 ms
  disconnect: 0.2 ms
  ingest wall (4a+4b): 6328.7 ms
  4a (10 singles) wall 297.6 ms  send+wait: min=293.6  median=294.2  max=294.6 ms
  4b (990 rows × 10 runs)  wall min=600.9  median=603.5  max=604.4 ms
  → appended to benchmark_results.jsonl


### Step 6: Results

Display all results from `benchmark_results.jsonl` as a flat table.
When the same `(mode, scenario)` appears more than once, numeric columns are
aggregated to the **median** and a `run_count` column shows how many runs were merged.

In [8]:
zbv2.display_results(_RESULTS_FILE)

,scenario,rows,runs,run_count,mode_label,concurrency,mode,n,oauth_ms,ping_ms,http_ping_ms,connect_ms,disconnect_ms,ingest_wall_ms,visibility_from_first_send_ms,visibility_from_end_ms,wall_ms,send_wait_ms_min,send_wait_ms_mean,send_wait_ms_median,send_wait_ms_max,send_ms_min,send_ms_mean,send_ms_median,send_ms_max,wait_ms_min,wait_ms_mean,wait_ms_median,wait_ms_max,table,zerobus_endpoint
0,4a,1,10,30,gRPC sync,1,grpc_sync,"1,000",NaN,92,NaN,"1,190",0,"4,287","7,912","3,329","2,010",193,201,201,207,0,0,0,0,193,201,201,207,main.robert_lee.airquality_grpc_sync,https://1444828305810485.zerobus.us-west-2.clo...
1,4b,990,10,30,gRPC sync,1,grpc_sync,"1,000",NaN,92,NaN,"1,190",0,"4,287","7,912","3,329","2,214",197,221,202,403,3,5,5,8,190,217,197,399,main.robert_lee.airquality_grpc_sync,https://1444828305810485.zerobus.us-west-2.clo...
2,4a,1,10,20,gRPC async,1,grpc_async,"1,000",NaN,146,NaN,"1,157",0,"4,328","6,827","2,543","1,977",159,198,201,209,0,0,0,0,159,197,201,209,main.robert_lee.airquality_grpc_async,https://1444828305810485.zerobus.us-west-2.clo...
3,4b,990,10,20,gRPC async,1,grpc_async,"1,000",NaN,146,NaN,"1,157",0,"4,328","6,827","2,543","2,311",195,231,202,404,3,5,5,8,190,224,197,399,main.robert_lee.airquality_grpc_async,https://1444828305810485.zerobus.us-west-2.clo...
4,4a,1,10,20,HTTP/1.1 sync,1,http_sync,"1,000",439,173,82,358,1,"4,188","6,553","2,331","2,089",194,209,201,277,194,209,201,277,0,0,0,0,main.robert_lee.airquality_http_sync,https://1444828305810485.zerobus.us-west-2.clo...
5,4b,990,10,20,HTTP/1.1 sync,1,http_sync,"1,000",439,173,82,358,1,"4,188","6,553","2,331","2,012",195,201,200,212,195,201,200,212,0,0,0,0,main.robert_lee.airquality_http_sync,https://1444828305810485.zerobus.us-west-2.clo...
6,4a,1,10,20,HTTP/1.1 async,1,http_async,"1,000",460,169,79,332,1,"4,148","7,001","2,567","2,103",193,210,201,289,193,210,201,289,0,0,0,0,main.robert_lee.airquality_http_async,https://1444828305810485.zerobus.us-west-2.clo...
7,4b,990,10,20,HTTP/1.1 async,1,http_async,"1,000",460,169,79,332,1,"4,148","7,001","2,567","2,014",196,201,200,209,196,201,200,209,0,0,0,0,main.robert_lee.airquality_http_async,https://1444828305810485.zerobus.us-west-2.clo...
8,4a,1,10,20,HTTP/2 sync,1,http2_sync,"1,000",426,160,82,336,0,"4,152","7,037","2,668","2,091",196,209,201,283,196,209,201,283,0,0,0,0,main.robert_lee.airquality_http2_sync,https://1444828305810485.zerobus.us-west-2.clo...
9,4b,990,10,20,HTTP/2 sync,1,http2_sync,"1,000",426,160,82,336,0,"4,152","7,037","2,668","2,009",196,201,200,205,196,201,200,205,0,0,0,0,main.robert_lee.airquality_http2_sync,https://1444828305810485.zerobus.us-west-2.clo...


### Step 7: Export to CSV

In [9]:
import os
_csv_path = Path(os.path.dirname(os.path.abspath(__vsc_ipynb_file__)) if "__vsc_ipynb_file__" in vars() else ".") / "benchmark_results.csv"
_df = zbv2.flatten_jsonl_to_df(_RESULTS_FILE)
zbv2.write_csv_from_df(_df, _csv_path)

Wrote CSV → /Users/robert.lee/github/zerobusdemo/notebooks/benchmark_results.csv  (714 rows)
